# 20 Preguntas de Negocio

In [1]:
# PostgreSQL connection
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os

load_dotenv()

def connect_to_db():
    try:
        connection = psycopg2.connect(
            dbname=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT")
        )
        print("Connection to PostgreSQL database successful")
        return connection
    except Exception as e:
        print(f"Error connecting to PostgreSQL database: {e}")
        return None

In [ ]:
# Count rows per year in the taxi_trips table
def count_rows_per_year(connection):
    try:
        cursor = connection.cursor()
        query = sql.SQL("""
            SELECT LEFT(source_month, 4) AS year, COUNT(*) AS trip_count
            FROM raw.raw_taxi_trips
            GROUP BY year
            ORDER BY year;
        """)
        cursor.execute(query)
        results = cursor.fetchall()
        print("Number of rows per year in the raw.raw_taxi_trips table:")
        for row in results:
            print(f"Year: {int(row[0])}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error counting rows per year in the raw.raw_taxi_trips table: {e}")
    finally:
        cursor.close()

if __name__ == "__main__":
    connection = connect_to_db()
    if connection:
        count_rows_per_year(connection)
        connection.close()

Connection to PostgreSQL database successful
Number of rows per year in the raw.raw_taxi_trips table:
Year: 2022, Trip Count: 40496500
Year: 2023, Trip Count: 39097286
Year: 2024, Trip Count: 41829938
Year: 2025, Trip Count: 44960735


In [ ]:
# Check missing months per year in the taxi_trips table
def check_missing_months_per_year(connection):
    try:
        cursor = connection.cursor()
        query = sql.SQL("""
            WITH months AS (
                SELECT DISTINCT LEFT(source_month, 4) AS year, RIGHT(source_month, 2) AS month
                FROM raw.raw_taxi_trips
            ),
            all_months AS (
                SELECT year, month
                FROM (SELECT DISTINCT LEFT(source_month, 4) AS year FROM raw.raw_taxi_trips) years,
                     (SELECT LPAD(generate_series(1, 12)::text, 2, '0') AS month) months
            )
            SELECT am.year, am.month
            FROM all_months am
            LEFT JOIN months m ON am.year = m.year AND am.month = m.month
            WHERE m.month IS NULL
            ORDER BY am.year, am.month;
        """)
        cursor.execute(query)
        results = cursor.fetchall()
        print("Missing months per year in the raw.raw_taxi_trips table:")
        for row in results:
            print(f"Year: {int(row[0])}, Missing Month: {row[1]}")
    except Exception as e:
        print(f"Error checking missing months per year in the raw.raw_taxi_trips table: {e}")
    finally:
        cursor.close()

if __name__ == "__main__":
    connection = connect_to_db()
    if connection:
        check_missing_months_per_year(connection)
        connection.close()

Connection to PostgreSQL database successful
Missing months per year in the bronze.stg_taxi_trips table:
Year: 2025, Missing Month: 12


In [3]:
# Get distinct values of payment types in the taxi_trips table
def get_distinct_payment_types(connection):
    try:
        cursor = connection.cursor()
        query = sql.SQL("""
            SELECT DISTINCT payment_type
            FROM raw.raw_taxi_trips
            ORDER BY payment_type;
        """)
        cursor.execute(query)
        results = cursor.fetchall()
        print("Distinct payment types in the raw.raw_taxi_trips table:")
        for row in results:
            print(f"Payment Type: {row[0]}")
    except Exception as e:
        print(f"Error getting distinct payment types in the raw.raw_taxi_trips table: {e}")
    finally:
        cursor.close()

if __name__ == "__main__":
    connection = connect_to_db()
    if connection:
        get_distinct_payment_types(connection)
        connection.close()

Connection to PostgreSQL database successful
Distinct payment types in the raw.raw_taxi_trips table:
Payment Type: 0.0
Payment Type: 1.0
Payment Type: 2.0
Payment Type: 3.0
Payment Type: 4.0
Payment Type: 5.0
Payment Type: None


### 1. Viajes por mes en 2024:

### 2. Viajes por *service_type* y mes

### 3. Top 10 zonas de pickup (total 2024)

### 4. Top 10 zonas de dropoff (total 2024)

### 5. Top 5 *boroughs* por mes (pickup)

### 6. Horas pico (top 5) por cada día de la semana

### 7. Distribución de viajes por día de la semana (ranking)

### 8. Ingreso total por mes

### 9. Ingreso total por *service_type* y mes

### 10. tip% promedio por mes

### 11. tip% por *borough* y mes

### 12. Top 10 zonas por ingreso total (pickup)

### 13. Top 10 zonas por tip% (pickup) con mínimo N viajes (N documentado)

### 14. Comparación *cash vs card*: viajes, ingreso total, tip%

### 15. Duración promedio (min) por mes

### 16. Distancia promedio por mes

### 17. Velocidad promedio (mph) por *borough* y franja horaria

### 18. Percentiles 50 y 90 de duración por borough

### 19. Top 10 zonas (pickup) por percentil 90 de duración

### 20. Top 10 rutas *borough to borough* por número de viajes